In [0]:
spark.conf.set(
    "fs.azure.account.key.saokrdataops001.dfs.core.windows.net",
    dbutils.secrets.get(scope="my-scope", key="storage"))

In [0]:
df = spark.read.json("abfss://examplecontainer@saokrdataops001.dfs.core.windows.net/mydataset.json")

In [0]:
import pyspark.sql.functions as F
participants = df.withColumn("participants", F.explode(F.col("Participants"))).select(F.col("participants.*"))
# participants.show()

In [0]:
courses = df.withColumn("courses", F.explode(F.col("Courses"))).select(F.col("courses.*")).withColumnRenamed("Name", "CourseName")
# courses.show()

Uitleggen van de functies

In [0]:
schoolclasses = participants.join(courses, "Project")
schoolclasses.show()

In [0]:
schoolclasses_count = schoolclasses.groupBy("CourseName").count()
schoolclasses_count.display()

In [0]:
schoolclasses.createOrReplaceTempView('schoolclasses')

In [0]:
spark.sql('describe table schoolclasses').show()

In [0]:
%sql
select * from schoolclasses;

In [0]:
schoolclasses_count.write.mode("overwrite").format("delta").save("abfss://examplecontainer@saokrdataops001.dfs.core.windows.net/schoolclasses_count")

In [0]:
spark.read.format("delta").load("abfss://examplecontainer@saokrdataops001.dfs.core.windows.net/schoolclasses_count").display()